In [8]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(project_root)

d:\Projects\data-lab


In [23]:
from datetime import date

from src.common.ch_client import ch_select, ch_execute
from src.pipelines.cbr_pipeline import get_cbr_rates_history
from src.load.clickhouse_load import (
    delete_currency_rates_by_source_date,
    load_currency_rates_to_clickhouse
)

In [27]:
import importlib
import src.load.clickhouse_load

importlib.reload(src.load.clickhouse_load)

from src.load.clickhouse_load import load_currency_rates_to_clickhouse

In [10]:
start_date = date(2026, 6, 10)
end_date = date(2026, 6, 16)

In [11]:
currency_rates_history_df = get_cbr_rates_history(
    start_date,
    end_date
)

In [12]:
ch_select(
    """
    describe table data_lab.raw_cbr_rates
    """
)

,name,type,default_type,default_expression,comment,codec_expression,ttl_expression
0,load_dttm,DateTime,,,,,
1,sourse_date,Date,,,,,
2,source_date,Date,,,,,
3,rate_date,Date,,,,,
4,currency_code,String,,,,,
5,currency_name,String,,,,,
6,nominal,UInt32,,,,,
7,rate,Float64,,,,,


In [13]:
ch_execute(
    """
    alter table data_lab.raw_cbr_rates
    add column if not exists sourse_date Date
    after load_dttm
"""
)

In [14]:
delete_result = delete_currency_rates_by_source_date(
    start_date,
    end_date
)

delete_result

{'table_name': 'data_lab.raw_cbr_rates',
 'start_date': datetime.date(2026, 6, 10),
 'end_date': datetime.date(2026, 6, 16),
 'status': 'success'}

In [15]:
load_result = load_currency_rates_to_clickhouse(
    currency_rates_history_df
)

load_result

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 378,
 'status': 'success'}

In [25]:
row_count_df = ch_select("""
    SELECT
        count(*) AS row_count
    FROM data_lab.raw_cbr_rates
""")

row_count_df

,row_count
0,1512


In [26]:
duplicate_check_df = ch_select("""
    SELECT
        source_date,
        currency_code,
        count(*) AS row_count
    FROM data_lab.raw_cbr_rates
    GROUP BY
        source_date,
        currency_code
    HAVING row_count > 1
    ORDER BY
        source_date,
        currency_code
""")

duplicate_check_df

,source_date,currency_code,row_count
0,2026-06-10,AED,4
1,2026-06-10,AMD,4
2,2026-06-10,AUD,4
3,2026-06-10,AZN,4
4,2026-06-10,BDT,4
...,...,...,...
373,2026-06-16,USD,4
374,2026-06-16,UZS,4
375,2026-06-16,VND,4
376,2026-06-16,XDR,4


In [18]:
from src.quality.cbr_checks import validate_cbr_rates_df

In [19]:
validation_result = validate_cbr_rates_df(
    currency_rates_history_df
)

validation_result

{'status': 'success', 'row_count': 378, 'errors': []}

In [20]:
load_result = load_currency_rates_to_clickhouse(
    currency_rates_history_df
)

load_result

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 378,
 'status': 'success'}

In [21]:
broken_currency_rates_df = currency_rates_history_df.copy()

broken_currency_rates_df.loc[
    0,
    'rate'
] = -1

In [28]:
load_currency_rates_to_clickhouse(
    broken_currency_rates_df
)

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 0,
 'status': 'failed',
 'validation_errors': ['Invalid rate values: 1']}